In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.hip)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.8.0+gitb2fb688
True
7.0.51831-a3e329ad8



In [ ]:
import torch

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0))

In [2]:
pip install ultralytics


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
# Define the contents of the new dataset.yaml
import os

# Detect environment: Kaggle or local
if os.path.exists('/kaggle/input'):
    dataset_base = '/kaggle/input/ppe-detection-m/final_merged_dataset'
else:
    # For local testing, use a placeholder path (dataset should be created first)
    dataset_base = './dataset_root'

yaml_content = f"""
path: {dataset_base}

train: train/images
val: valid/images
test: test/images

nc: 8

names:
  - no-safety-glove
  - no-safety-helmet
  - no-safety-shoes
  - no-welding-glass
  - safety-glove
  - safety-helmet
  - safety-shoes
  - welding-glass
augmentation:
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  degrees: 0.0
  translate: 0.1
  scale: 0.5
  shear: 0.0
  perspective: 0.0
  flipud: 0.0
  fliplr: 0.5  
"""

# Save to current directory
with open('dataset.yaml', 'w') as f:
    f.write(yaml_content.strip())

In [1]:
!pip install -q ultralytics



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [ ]:
import os
import subprocess
import zipfile
from pathlib import Path


# Setup paths
dataset_dir = Path('./dataset_root')
dataset_dir.mkdir(exist_ok=True)

# Check if dataset already exists
if (dataset_dir / 'train' / 'images').exists():
    print("✅ Dataset already available!")
    train_images = len(list((dataset_dir / 'train' / 'images').glob('*.jpg')))
    train_labels = len(list((dataset_dir / 'train' / 'labels').glob('*.txt')))
    valid_images = len(list((dataset_dir / 'valid' / 'images').glob('*.jpg')))
    valid_labels = len(list((dataset_dir / 'valid' / 'labels').glob('*.txt')))
    
    print(f"\n📊 Dataset Summary:")
    print(f"   📂 Train: {train_images} images, {train_labels} labels")
    print(f"   📂 Valid: {valid_images} images, {valid_labels} labels")
    print(f"   📂 Location: {dataset_dir.resolve()}")
    print(f"\n🎉 Ready to train! Proceed to the Training section.")
else:
    print("📥 Starting dataset download via Kaggle API...")
    print("⏳ This may take a few minutes depending on your internet speed...\n")
    
    try:
        # Download using Kaggle API
        dataset_name = "anuragraj03/ppe-detection-m"
        result = subprocess.run(
            f"kaggle datasets download -d {dataset_name} -p {dataset_dir}",
            shell=True,
            capture_output=True,
            text=True,
            timeout=600  # 10 minute timeout
        )
        
        if result.returncode == 0:
            print("✅ Download successful!")
            
            # Extract zip files
            zip_files = list(dataset_dir.glob('*.zip'))
            if zip_files:
                print(f"📦 Extracting {len(zip_files)} file(s)...")
                for zip_file in zip_files:
                    print(f"   📂 {zip_file.name}")
                    try:
                        with zipfile.ZipFile(zip_file, 'r') as zf:
                            zf.extractall(dataset_dir)
                        zip_file.unlink()  # Remove zip after extraction
                    except Exception as e:
                        print(f"   ⚠️  Error extracting {zip_file.name}: {e}")
                
                print("✅ Extraction complete!")
                
                # Flatten structure if needed
                nested_dir = dataset_dir / 'final_merged_dataset'
                if nested_dir.exists():
                    print("📂 Reorganizing dataset structure...")
                    for item in nested_dir.iterdir():
                        if item.is_dir():
                            from shutil import copytree
                            target = dataset_dir / item.name
                            if target.exists():
                                from shutil import rmtree
                                rmtree(target)
                            copytree(item, target)
                        else:
                            import shutil
                            shutil.copy2(item, dataset_dir / item.name)
                    from shutil import rmtree
                    rmtree(nested_dir)
                    print("✅ Structure reorganized!")
                
                # Verify structure
                train_images = dataset_dir / 'train' / 'images'
                train_labels = dataset_dir / 'train' / 'labels'
                valid_images = dataset_dir / 'valid' / 'images'
                valid_labels = dataset_dir / 'valid' / 'labels'
                
                if all([train_images.exists(), train_labels.exists(), 
                        valid_images.exists(), valid_labels.exists()]):
                    train_img_count = len(list(train_images.glob('*.jpg')))
                    train_lbl_count = len(list(train_labels.glob('*.txt')))
                    valid_img_count = len(list(valid_images.glob('*.jpg')))
                    valid_lbl_count = len(list(valid_labels.glob('*.txt')))
                    
                    print("\n✅ Dataset structure verified!")
                    print(f"   📂 Train images: {train_img_count} files")
                    print(f"   📂 Train labels: {train_lbl_count} files")
                    print(f"   📂 Valid images: {valid_img_count} files")
                    print(f"   📂 Valid labels: {valid_lbl_count} files")
                    print(f"\n🎉 Ready to train! Dataset at: {dataset_dir.resolve()}")
                else:
                    print("\n⚠️  Dataset structure incomplete. Expected directories not found.")
            else:
                print("⚠️  No zip files found to extract.")
        else:
            print(f"❌ Download failed!")
            print(f"Error: {result.stderr}")
            if "authentication" in result.stderr.lower():
                print("\n🔐 Authentication Issue:")
                print("   1. Go to: https://www.kaggle.com/settings/api")
                print("   2. Click 'Generate New Token'")
                print("   3. Save kaggle.json to ~/.kaggle/")
                print("   4. Run this cell again")
            
    except subprocess.TimeoutExpired:
        print("❌ Download timed out. Try again or download manually from:")
        print("   https://www.kaggle.com/datasets/anuragraj03/ppe-detection-m")
    except FileNotFoundError:
        print("❌ Kaggle CLI not found. Installing...")
        subprocess.run("pip install kaggle -q", shell=True)
        print("✅ Kaggle installed. Please re-run this cell after setting up credentials.")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")

📥 Starting dataset download via Kaggle API...
⏳ This may take a few minutes depending on your internet speed...

✅ Download successful!
📦 Extracting 1 file(s)...
   📂 ppe-detection-m.zip
✅ Extraction complete!
📂 Reorganizing dataset structure...


In [12]:
!pip install kaggle


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [kaggle]2m5/6 [kaggle]dk]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [15]:
!kaggle --version

Kaggle CLI 2.2.1


In [1]:
custom_aug_code = """
import cv2
import numpy as np
import random

def get_class_specific_augmentation(class_name):
    '''Returns augmentation function for a given class using OpenCV'''
    
    def apply_aug(image, bboxes, class_labels):
        result_image = image.copy()
        result_bboxes = bboxes.copy()
        
        # Basic augmentations using OpenCV
        if 'safety-shoes' in class_name.lower():
            # Random brightness/contrast for shoes
            if random.random() > 0.2:
                result_image = cv2.convertScaleAbs(result_image, alpha=1.2, beta=30)
            # Random blur to simulate motion
            if random.random() > 0.7:
                result_image = cv2.GaussianBlur(result_image, (5, 5), 0)
        
        elif 'safety-glove' in class_name.lower():
            # Color shift for gloves
            if random.random() > 0.3:
                hsv = cv2.cvtColor(result_image, cv2.COLOR_BGR2HSV)
                hsv[:, :, 0] = np.clip(hsv[:, :, 0] + random.randint(-20, 20), 0, 179)
                result_image = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
            # Random rotation
            if random.random() > 0.5:
                h, w = result_image.shape[:2]
                angle = random.randint(-15, 15)
                M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
                result_image = cv2.warpAffine(result_image, M, (w, h))
        
        elif 'safety-helmet' in class_name.lower():
            # Brightness adjustment
            if random.random() > 0.5:
                result_image = cv2.convertScaleAbs(result_image, alpha=1.1, beta=20)
            # Slight rotation
            if random.random() > 0.5:
                h, w = result_image.shape[:2]
                angle = random.randint(-10, 10)
                M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
                result_image = cv2.warpAffine(result_image, M, (w, h))
        
        else:
            # General augmentation
            if random.random() > 0.5:
                result_image = cv2.convertScaleAbs(result_image, alpha=1.1, beta=15)
        
        return result_image, result_bboxes, class_labels
    
    return apply_aug
"""

with open('custom_augmentation.py', 'w') as f:
    f.write(custom_aug_code)

In [3]:
apply_aug_code = """
import cv2
import numpy as np
from pathlib import Path
from custom_augmentation import get_class_specific_augmentation

# Must match dataset.yaml order
class_names = [
    'no-safety-glove', 'no-safety-helmet', 'no-safety-shoes', 'no-welding-glass',
    'safety-glove', 'safety-helmet', 'safety-shoes', 'welding-glass'
]

def apply_augmentation(image_path: Path, label_path: Path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    h, w = image.shape[:2]

    with open(label_path, 'r') as f:
        lines = f.read().strip().split('\\n')

    bboxes = []
    class_ids = []
    for line in lines:
        parts = line.strip().split()
        cid = int(parts[0])
        x, y, bw, bh = map(float, parts[1:])
        # Convert from YOLO to COCO format
        x1 = (x - bw / 2) * w
        y1 = (y - bh / 2) * h
        bboxes.append([x1, y1, bw * w, bh * h])
        class_ids.append(cid)

    aug_func = get_class_specific_augmentation(class_names[class_ids[0]] if class_ids else 'unknown')
    aug_image, aug_bboxes, aug_labels = aug_func(image, bboxes, class_ids)

    return aug_image, aug_bboxes, aug_labels
"""

with open('apply_augmentation_script.py', 'w') as f:
    f.write(apply_aug_code)

In [5]:
import os
import cv2
import shutil
from pathlib import Path
from tqdm import tqdm
from apply_augmentation_script import apply_augmentation

# Paths
dataset_dir = Path('/kaggle/input/ppe-detection-m/final_merged_dataset/train')
aug_output_dir = Path('/kaggle/working/augmented/train')

# Create directories for augmented images and labels
(aug_output_dir / 'images').mkdir(parents=True, exist_ok=True)
(aug_output_dir / 'labels').mkdir(parents=True, exist_ok=True)

# Loop through all label files
label_dir = dataset_dir / 'labels'
image_dir = dataset_dir / 'images'

image_files = sorted([f for f in image_dir.glob('*.jpg')])
augmented_count = 0

for image_path in tqdm(image_files, desc="🔁 Augmenting weak classes"):
    label_path = label_dir / f"{image_path.stem}.txt"
    if not label_path.exists():
        continue

    try:
        aug_image, aug_bboxes, aug_labels = apply_augmentation(image_path, label_path)
    except Exception as e:
        continue  # Skip any problematic file

    # Only save if augmentation was applied (e.g. for weak classes)
    if len(aug_bboxes) > 0:
        h, w = aug_image.shape[:2]
        aug_label_lines = []

        for bbox, cls in zip(aug_bboxes, aug_labels):
            # Convert back from COCO to YOLO format
            x, y, bw, bh = bbox
            x_center = (x + bw / 2) / w
            y_center = (y + bh / 2) / h
            bw_yolo = bw / w
            bh_yolo = bh / h
            aug_label_lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {bw_yolo:.6f} {bh_yolo:.6f}")

        # Save image
        aug_img_path = aug_output_dir / 'images' / f"{image_path.stem}_aug.jpg"
        aug_label_path = aug_output_dir / 'labels' / f"{image_path.stem}_aug.txt"
        cv2.imwrite(str(aug_img_path), aug_image)

        # Save label
        with open(aug_label_path, 'w') as f:
            f.write('\n'.join(aug_label_lines))

        augmented_count += 1

print(f"✅ Done. Augmented {augmented_count} weak-class images.")


🔁 Augmenting weak classes: 0it [00:00, ?it/s]

✅ Done. Augmented 0 weak-class images.


In [7]:
import os
from pathlib import Path

# Check if we're on Kaggle or local environment
on_kaggle = os.path.exists('/kaggle/input')

if on_kaggle:
    dataset_path = Path('/kaggle/input/ppe-detection-m/final_merged_dataset')
    print("✅ Running on Kaggle environment")
else:
    dataset_path = Path('./dataset_root')
    print("⚠️  Running on local environment")
    if not dataset_path.exists():
        print(f"\n❌ Dataset not found at: {dataset_path}")
        print("\n📋 To run training locally, you need:")
        print("   1. Download the PPE Detection dataset from Kaggle")
        print("   2. Extract it to './dataset_root/' directory")
        print("   3. Ensure the structure is:")
        print("      - dataset_root/train/images/")
        print("      - dataset_root/train/labels/")
        print("      - dataset_root/valid/images/")
        print("      - dataset_root/valid/labels/")
        print("      - dataset_root/test/images/")
        print("      - dataset_root/test/labels/")
        print("\n🔗 Get dataset from: https://www.kaggle.com/datasets/...")
    else:
        print(f"✅ Dataset found at: {dataset_path}")

# Verify YOLO model is ready
if os.path.exists('yolov9c.pt'):
    print("✅ YOLOv9c model weights ready")
else:
    print("⚠️  YOLOv9c will be downloaded during training...")


⚠️  Running on local environment
✅ Dataset found at: dataset_root
⚠️  YOLOv9c will be downloaded during training...


In [2]:
from ultralytics import YOLO
import torch
import os
from pathlib import Path

# Load model
model = YOLO("yolov8m.pt")

# Check dataset existence before training
on_kaggle = os.path.exists('/kaggle/input')
if on_kaggle:
    dataset_path = "./dataset.yaml"
    data_dir = Path('./datasets')
else:
    dataset_path = "./dataset.yaml"
    data_dir = Path("./datasets")

if not data_dir.exists():
    print(f"⚠️  Dataset directory not found: {data_dir}")
    print("\nℹ️  This notebook requires the PPE Detection dataset.")
    print("On Kaggle: Dataset will be auto-loaded from /kaggle/input/")
    print("Locally: Download the dataset and extract to './dataset_root/'")
    print("\n✅ Fix: Download dataset and retry training")
else:
    print(f"✅ Dataset ready at: {data_dir}")
    print("\n🚀 Starting YOLOv8 training...")
    
  
    
    results = model.train(
    data="dataset.yaml",
    epochs=30,        # or 50 for demo
    batch=64,
    imgsz=640,
    device=0,
    amp=False,
    val=False,        # skip validation
    classes=[4,5,6],
    optimizer="Adam",
    lr0=0.001,
    lrf=0.01,
    patience=20,
    workers=16,
    cache=True,
    project="runs/transfer_learning",
    name="yolov8_ppe_mi300x",
    exist_ok=True
)

    print("✅ Training Completed")
    


✅ Dataset ready at: datasets

🚀 Starting YOLOv8 training...
Ultralytics 8.4.66 🚀 Python-3.12.11 torch-2.8.0+gitb2fb688 CUDA:0 (, 196592MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=True, cfg=None, classes=[4, 5, 6], close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_ppe_mi300x, nbs=64, nms=False, opset=None, optimize=F

NotImplementedError: Could not run 'torchvision::nms' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'torchvision::nms' is only available for these backends: [CPU, Meta, QuantizedCPU, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradMeta, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFrontMode, PreDispatch, PythonDispatcher].

CPU:

In [3]:
from pathlib import Path

for f in Path("runs").rglob("*.pt"):
    print(f)

runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/last.pt
runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt


In [6]:
from ultralytics import YOLO

model = YOLO(
    "runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt"
)


model.names

{0: 'no-safety-glove',
 1: 'no-safety-helmet',
 2: 'no-safety-shoes',
 3: 'no-welding-glass',
 4: 'safety-glove',
 5: 'safety-helmet',
 6: 'safety-shoes',
 7: 'welding-glass'}

In [7]:
import torch
import torchvision

boxes = torch.tensor([
    [0,0,10,10],
    [1,1,11,11]
], device="cuda")

scores = torch.tensor([0.9,0.8], device="cuda")

torchvision.ops.nms(boxes, scores, 0.5)

NotImplementedError: Could not run 'torchvision::nms' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'torchvision::nms' is only available for these backends: [CPU, Meta, QuantizedCPU, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradMeta, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFrontMode, PreDispatch, PythonDispatcher].

CPU:

In [9]:
from ultralytics import YOLO

model = YOLO(
    "runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt"
)

results = model.predict(
    source="/workspace/shared/datasets/final_merged_dataset/test/images",
    device="cpu",
    conf=0.25,
    save=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2276 /workspace/shared/datasets/final_merged_dataset/test/images/005298_jpg.rf.eff75fb220fd3090ecc71ab3daca1709.jpg: 448x640 (no detections), 115.8ms
image 2/2276 /workspace/shared/datasets/final_merged_dataset/test/images/005299_jpg.rf.2950c47dea579180ab7b0ae70900fa3f.jpg: 448x640 (no detections), 83.2ms
image 3/2276 /workspace/shared/datasets/final_merged_dataset/test/images/005300_jpg.rf.da8929edc3745e8545144cfff8c6a0ef.jpg: 480x640 (no detect

KeyboardInterrupt: 

In [11]:
from ultralytics import YOLO

model = YOLO(
    "runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt"
)

results = model.predict(
    source="/workspace/shared/datasets/test_image.jpg",
    device="cpu",
    conf=0.25,
    save=True
)


image 1/1 /workspace/shared/datasets/test_image.jpg: 480x640 (no detections), 105.9ms
Speed: 1.3ms preprocess, 105.9ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)
Results saved to /workspace/shared/runs/detect/predict-5


In [12]:
import subprocess
import time
import pandas as pd
import psutil  # for CPU and RAM usage

log_file = "gpu_metrics_final.csv"

# Write CSV header
with open(log_file, "w") as f:
    f.write(
        "timestamp,gpu_utilization(%),mem_used(MB),mem_total(MB),mem_utilization(%),"
        "power(W),temp(C),gpu_clock(MHz),mem_clock(MHz),cpu_util(%),ram_used(GB)\n"
    )

for _ in range(100):  # run while training
    timestamp = time.time()
    
    # Get GPU metrics from ROCm
    output = subprocess.check_output([
        "rocm-smi",
        "--showuse",
        "--showmemuse",
        "--showpower",
        "--showtemp",
        "--showclocks",
        "--csv"
    ]).decode()
    
    lines = output.strip().split("\n")[1:]  # skip header
    
    # CPU and RAM usage
    cpu_util = psutil.cpu_percent()
    ram_used = psutil.virtual_memory().used / (1024 ** 3)  # GB

    for line in lines:
        parts = line.split(",")
        gpu_util = float(parts[1].strip())
        
        mem_used = float(parts[2].strip().split("/")[0])
        mem_total = float(parts[2].strip().split("/")[1])
        mem_util = (mem_used / mem_total) * 100
        
        power = float(parts[3].strip())
        temp = float(parts[4].strip())
        
        gpu_clock = parts[5].strip() if len(parts) > 5 else "N/A"
        mem_clock = parts[6].strip() if len(parts) > 6 else "N/A"
        
        with open(log_file, "a") as f:
            f.write(
                f"{timestamp},{gpu_util},{mem_used},{mem_total},{mem_util},"
                f"{power},{temp},{gpu_clock},{mem_clock},{cpu_util},{ram_used}\n"
            )
    
    time.sleep(1)

IndexError: list index out of range

In [14]:
from ultralytics import YOLO
import json

# Load trained model
model = YOLO(
    "runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt"
)

# Run inference
results = model.predict(
    source="/workspace/shared/datasets/test_image.jpg",   # replace with your image path
    device="cpu",
    conf=0.25
)

# Extract detections
detections = []

for result in results:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])

        x1, y1, x2, y2 = box.xyxy[0].tolist()

        detections.append({
            "class_id": cls_id,
            "class_name": model.names[cls_id],
            "confidence": round(conf, 4),
            "bbox": {
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "x2": round(x2, 2),
                "y2": round(y2, 2)
            }
        })

# Save JSON
with open("detections.json", "w") as f:
    json.dump(detections, f, indent=4)

print("✅ Saved detections.json")


image 1/1 /workspace/shared/datasets/test_image.jpg: 480x640 (no detections), 104.7ms
Speed: 1.3ms preprocess, 104.7ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)
✅ Saved detections.json


In [16]:
from ultralytics import YOLO

model = YOLO(
    "runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt"
)

results = model.predict(
    source="/workspace/shared/datasets/test_image.jpg",   # your image
    device="cpu",
    conf=0.25
)

print(results)


image 1/1 /workspace/shared/datasets/test_image.jpg: 480x640 (no detections), 104.1ms
Speed: 1.4ms preprocess, 104.1ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)
[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'no-safety-glove', 1: 'no-safety-helmet', 2: 'no-safety-shoes', 3: 'no-welding-glass', 4: 'safety-glove', 5: 'safety-helmet', 6: 'safety-shoes', 7: 'welding-glass'}
obb: None
orig_img: array([[[197, 148, 104],
        [197, 148, 104],
        [197, 148, 104],
        ...,
        [ 87,  88,  56],
        [105, 101,  66],
        [123, 116,  77]],

       [[197, 148, 104],
        [197, 148, 104],
        [198, 149, 105],
        ...,
        [ 89,  89,  59],
        [106, 102,  67],
        [124, 117,  78]],

       [[198, 149, 105],
        [198, 149, 105],
        [198, 149, 105],
        ...,
        [ 90,  92,  62],
        [107, 103,  68],
        [125, 1

In [22]:
from ultralytics import YOLO
import json

# Load model
model = YOLO(
    "runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt"
)

# Run inference
results = model.predict(
    source="/workspace/shared/datasets/test_image2.jpg",
    device="cpu",
    conf=0.25
)
print(model.names)


image 1/1 /workspace/shared/datasets/test_image2.jpg: 384x640 (no detections), 92.0ms
Speed: 1.2ms preprocess, 92.0ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
{0: 'no-safety-glove', 1: 'no-safety-helmet', 2: 'no-safety-shoes', 3: 'no-welding-glass', 4: 'safety-glove', 5: 'safety-helmet', 6: 'safety-shoes', 7: 'welding-glass'}


In [6]:
from ultralytics import YOLO
import json

# Load model
model = YOLO(
    "runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt"
)

# Inference
results = model.predict(
    source="/workspace/shared/datasets/test_image4.jpg",
    device="cpu",
    conf=0.01,
    imgsz=640
)
print("Predictions saved to:", results[0].save_dir)

# PPE categories we care about
ppe_items = ["glove", "helmet", "shoes", "welding-glass"]

# Initialize status
ppe_status = {item: False for item in ppe_items}

detections = []

for result in results:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        cls_name = model.names[cls_id]
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        # Update PPE status: detect "safety-*" classes
        for item in ppe_items:
            if cls_name == f"safety-{item}":
                ppe_status[item] = True

        detections.append({
            "class": cls_name,
            "confidence": round(conf, 3),
            "bbox": {
                "x1": round(x1,2),
                "y1": round(y1,2),
                "x2": round(x2,2),
                "y2": round(y2,2)
            }
        })

# Missing PPE
missing_items = [item for item, present in ppe_status.items() if not present]

output = {
    "image": "test_image2.jpg",
    "ppe_status": ppe_status,
    "missing_ppe": missing_items,
    "compliant": len(missing_items) == 0,
    "detections": detections
}

# Save JSON
with open("ppe_report.json", "w") as f:
    json.dump(output, f, indent=4)

# Preview JSON
print(json.dumps(output, indent=4))


image 1/1 /workspace/shared/datasets/test_image4.jpg: 640x640 13 safety-gloves, 3 safety-helmets, 7 safety-shoess, 118.1ms
Speed: 1.1ms preprocess, 118.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)
Predictions saved to: /workspace/shared/runs/detect/predict-6
{
    "image": "test_image2.jpg",
    "ppe_status": {
        "glove": true,
        "helmet": true,
        "shoes": true,
        "welding-glass": false
    },
    "missing_ppe": [
        "welding-glass"
    ],
    "compliant": false,
    "detections": [
        {
            "class": "safety-helmet",
            "confidence": 0.859,
            "bbox": {
                "x1": 111.1,
                "y1": 175.54,
                "x2": 148.28,
                "y2": 243.21
            }
        },
        {
            "class": "safety-shoes",
            "confidence": 0.849,
            "bbox": {
                "x1": 109.74,
                "y1": 338.0,
                "x2": 135.9,
                "y2": 

In [6]:
from ultralytics import YOLO
import json

# Image path (can replace with any single image)
IMAGE_PATH = "/workspace/shared/datasets/test_image5.jpg"

# Load models
person_model = YOLO("yolov8n.pt")  # Pretrained person detector
ppe_model = YOLO("runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt")  # Your trained PPE model

# 1️⃣ Detect persons
person_results = person_model.predict(
    source=IMAGE_PATH,
    device="cpu",
    conf=0.25,
    imgsz=640
)

persons = []
for i, box in enumerate(person_results[0].boxes):
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    persons.append({
        "person_id": i+1,
        "bbox": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
        "helmet": False,
        "glove": False,
        "shoes": False,
        "welding_glass": False
    })

# 2️⃣ Detect PPE items
ppe_results = ppe_model.predict(
    source=IMAGE_PATH,
    device="cpu",
    conf=0.05,
    imgsz=640
)

ppe_classes = ["safety-helmet", "safety-glove", "safety-shoes", "welding-glass"]
ppe_detections = []

# Map PPE class to person
for result in ppe_results:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        cls_name = ppe_model.names[cls_id]
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        # Save detection
        ppe_detections.append({
            "class": cls_name,
            "confidence": round(conf,3),
            "bbox": {"x1": round(x1,2),"y1": round(y1,2),"x2": round(x2,2),"y2": round(y2,2)}
        })

        # Assign to person if PPE overlaps
        for person in persons:
            px1, py1, px2, py2 = person["bbox"].values()
            if (x1 < px2 and x2 > px1 and y1 < py2 and y2 > py1):
                if cls_name == "safety-helmet":
                    person["helmet"] = True
                elif cls_name == "safety-glove":
                    person["glove"] = True
                elif cls_name == "safety-shoes":
                    person["shoes"] = True
                elif cls_name == "welding-glass":
                    person["welding_glass"] = True

# 3️⃣ Generate report
violations = []
for p in persons:
    missing = [k for k,v in p.items() if k in ["helmet","glove","shoes","welding_glass"] and not v]
    if missing:
        violations.append({"person_id": p["person_id"], "missing_ppe": missing})

report = {
    "image": IMAGE_PATH.split("/")[-1],
    "total_persons": len(persons),
    "persons_without_helmet": [p["person_id"] for p in persons if not p["helmet"]],
    "persons_without_glove": [p["person_id"] for p in persons if not p["glove"]],
    "persons_without_shoes": [p["person_id"] for p in persons if not p["shoes"]],
    "persons_without_welding_glass": [p["person_id"] for p in persons if not p["welding_glass"]],
    "violations": violations,
    "compliant": len(violations) == 0,
    "ppe_detections": ppe_detections
}

# Save JSON
with open("full_ppe_person_report.json", "w") as f:
    json.dump(report, f, indent=4)

# Preview
print(json.dumps(report, indent=4))
print("\n✅ Full PPE & person report saved as full_ppe_person_report.json")


image 1/1 /workspace/shared/datasets/test_image5.jpg: 320x640 5 persons, 1 truck, 3 suitcases, 23.1ms
Speed: 1.2ms preprocess, 23.1ms inference, 0.5ms postprocess per image at shape (1, 3, 320, 640)

image 1/1 /workspace/shared/datasets/test_image5.jpg: 320x640 (no detections), 89.8ms
Speed: 1.2ms preprocess, 89.8ms inference, 0.3ms postprocess per image at shape (1, 3, 320, 640)
{
    "image": "test_image5.jpg",
    "total_persons": 9,
    "persons_without_helmet": [
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        9
    ],
    "persons_without_glove": [
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        9
    ],
    "persons_without_shoes": [
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        9
    ],
    "persons_without_welding_glass": [
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        9
    ],
    "

In [7]:
!pip install -q ultralytics


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [11]:
from ultralytics import YOLO
import cv2

# Image path
IMAGE_PATH = "/workspace/shared/datasets/test_image3.jpg"
OUTPUT_PATH = "/workspace/shared/datasets/test_image3_annotated.jpg"

# Load models
person_model = YOLO("yolov8n.pt")  # Person detector
ppe_model = YOLO("runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt")  # Your PPE model

# Load image
img = cv2.imread(IMAGE_PATH)
img_draw = img.copy()

# 1️⃣ Detect persons
person_results = person_model.predict(
    source=img,
    device="cpu",
    conf=0.25,
    imgsz=640
)

persons = []
for i, box in enumerate(person_results[0].boxes):
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    persons.append({
        "person_id": i+1,
        "bbox": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
        "helmet": False,
        "glove": False,
        "shoes": False,
        "welding_glass": False
    })

# 2️⃣ Detect PPE items
ppe_results = ppe_model.predict(
    source=img,
    device="cpu",
    conf=0.05,
    imgsz=640
)

for result in ppe_results:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        cls_name = ppe_model.names[cls_id]
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        # Match PPE to person
        for person in persons:
            px1, py1, px2, py2 = person["bbox"].values()
            if (x1 < px2 and x2 > px1 and y1 < py2 and y2 > py1):
                if cls_name == "safety-helmet":
                    person["helmet"] = True
                elif cls_name == "safety-glove":
                    person["glove"] = True
                elif cls_name == "safety-shoes":
                    person["shoes"] = True
                elif cls_name == "welding-glass":
                    person["welding_glass"] = True

# 3️⃣ Draw bounding boxes and labels
for person in persons:
    x1, y1, x2, y2 = map(int, person["bbox"].values())
    cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red box

    # Build PPE missing text
    missing_items = []
    if not person["helmet"]:
        missing_items.append("no helmet")
    if not person["glove"]:
        missing_items.append("no glove")
    if not person["shoes"]:
        missing_items.append("no shoes")
    if not person["welding_glass"]:
        missing_items.append("no welding glass")

    label = ", ".join(missing_items)
    if label:
        cv2.putText(
            img_draw, label, (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2
        )

# 4️⃣ Save annotated image
cv2.imwrite(OUTPUT_PATH, img_draw)
print(f"✅ Annotated image saved at {OUTPUT_PATH}")


0: 384x640 3 persons, 2 hot dogs, 25.5ms
Speed: 1.0ms preprocess, 25.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 safety-gloves, 84.3ms
Speed: 1.1ms preprocess, 84.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
✅ Annotated image saved at /workspace/shared/datasets/test_image3_annotated.jpg


In [12]:
from ultralytics import YOLO
import cv2

IMAGE_PATH = "/workspace/shared/datasets/test_image3.jpg"
OUTPUT_PATH = "/workspace/shared/datasets/test_image3_annotated_full.jpg"

# Load PPE model
ppe_model = YOLO("runs/detect/runs/transfer_learning/yolov8_ppe_mi300x/weights/best.pt")
img = cv2.imread(IMAGE_PATH)
img_draw = img.copy()

# Detect all PPE items
ppe_results = ppe_model.predict(source=img, device="cpu", conf=0.05, imgsz=640)
ppe_detections = []
ppe_classes = ["safety-helmet", "safety-glove", "safety-shoes", "welding-glass"]

# Collect all PPE detections
for result in ppe_results:
    for box in result.boxes:
        cls_id = int(box.cls[0])
        cls_name = ppe_model.names[cls_id]
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        ppe_detections.append({"class": cls_name, "bbox": (x1,y1,x2,y2)})

# Cluster PPE boxes into persons
persons = []
visited = [False]*len(ppe_detections)

for i, det in enumerate(ppe_detections):
    if visited[i]:
        continue
    x1, y1, x2, y2 = det["bbox"]
    group_boxes = [det["bbox"]]
    group_classes = [det["class"]]
    visited[i] = True

    # Merge nearby PPE boxes to form a person
    for j, other in enumerate(ppe_detections):
        if visited[j]:
            continue
        ox1, oy1, ox2, oy2 = other["bbox"]
        # Check overlap
        iou_x1 = max(x1, ox1)
        iou_y1 = max(y1, oy1)
        iou_x2 = min(x2, ox2)
        iou_y2 = min(y2, oy2)
        if iou_x1 < iou_x2 and iou_y1 < iou_y2:
            group_boxes.append(other["bbox"])
            group_classes.append(other["class"])
            visited[j] = True

    # Person bbox is union of group
    gx1 = min([b[0] for b in group_boxes])
    gy1 = min([b[1] for b in group_boxes])
    gx2 = max([b[2] for b in group_boxes])
    gy2 = max([b[3] for b in group_boxes])

    # Track which PPE items are present
    persons.append({
        "bbox": (gx1, gy1, gx2, gy2),
        "helmet": "safety-helmet" in group_classes,
        "glove": "safety-glove" in group_classes,
        "shoes": "safety-shoes" in group_classes,
        "welding_glass": "welding-glass" in group_classes
    })

# Draw annotated persons
for p in persons:
    x1, y1, x2, y2 = p["bbox"]
    cv2.rectangle(img_draw, (x1, y1), (x2, y2), (0,0,255), 2)
    missing = []
    if not p["helmet"]:
        missing.append("no helmet")
    if not p["glove"]:
        missing.append("no glove")
    if not p["shoes"]:
        missing.append("no shoes")
    if not p["welding_glass"]:
        missing.append("no welding glass")
    label = ", ".join(missing)
    if label:
        cv2.putText(img_draw, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

# Save annotated image
cv2.imwrite(OUTPUT_PATH, img_draw)
print(f"✅ Annotated image saved at {OUTPUT_PATH}")


0: 384x640 4 safety-gloves, 85.3ms
Speed: 1.2ms preprocess, 85.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
✅ Annotated image saved at /workspace/shared/datasets/test_image3_annotated_full.jpg
